<a href="https://colab.research.google.com/github/springboardmentor12458j/LiveMeetingSummarize/blob/varshini/module-5%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install core dependencies
!pip install streamlit pyngrok audiorecorder -q

# Install PyTorch (CPU version for Colab)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu -q

# Install Transformers for Whisper and BART models
!pip install transformers -q

# Install audio processing library
!pip install librosa -q

# Install soundfile (required by librosa for audio loading)
!pip install soundfile -q

# Install additional dependencies
!pip install numpy -q
!pip install pyngrok -q

In [ ]:
%%writefile model.py
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration, BartTokenizer, BartForConditionalGeneration
import streamlit as st
import os
import warnings


# Suppress warnings
warnings.filterwarnings('ignore')


# CRITICAL: Disable all GPU/MPS usage
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"


# Monkey-patch torch.load to force CPU
original_torch_load = torch.load
def patched_torch_load(*args, **kwargs):
    kwargs['map_location'] = 'cpu'
    return original_torch_load(*args, **kwargs)
torch.load = patched_torch_load


@st.cache_resource
def load_whisper_elements():
    print("Loading Whisper Model (with CPU enforcement)...")
    processor = WhisperProcessor.from_pretrained("openai/whisper-base")

    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")
    model = model.to('cpu')  # Explicitly move model to CPU

    model.config.forced_decoder_ids = None
    model.eval()
    print("Whisper model loaded successfully")
    return processor, model


@st.cache_resource
def load_summarizer_elements():
    print("Loading Summarizer Model (with CPU enforcement)...")
    tokenizer = BartTokenizer.from_pretrained("sshleifer/distilbart-cnn-12-6")

    model = BartForConditionalGeneration.from_pretrained("sshleifer/distilbart-cnn-12-6")
    model = model.to('cpu')  # Explicitly move model to CPU

    model.eval()
    print("Summarizer model loaded successfully")
    return tokenizer, model


class TranscriptionModel:
    def __init__(self):
        self.processor, self.model = load_whisper_elements()


    def transribe(self, audio_path):
        print(f"Transcribing {audio_path}...")
        try:
            # Load audio at 16k sample rate
            audio_array, _ = librosa.load(audio_path, sr=16000)
            duration = librosa.get_duration(y=audio_array, sr=16000)
            print(f"Audio loaded: {duration}s, shape: {audio_array.shape}")

            # Process inputs
            input_features = self.processor(
                audio_array,
                sampling_rate=16000,
                return_tensors="pt"
            ).input_features


            # Generate tokens
            print("Generating tokens...")
            with torch.no_grad():
                predicted_ids = self.model.generate(input_features)
            print(f"Tokens generated: {predicted_ids.shape}")

            # Decode
            transcription = self.processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
            print(f"Transcribed Text: '{transcription}'")

            final_text = transcription.strip()
            if not final_text:
                final_text = "[Audio detected but no speech recognized]"

            return [{
                "start": 0.0,
                "end": duration,
                "text": final_text
            }]
        except Exception as e:
            print(f"Transcription Error: {e}")
            import traceback
            traceback.print_exc()
            return [{
                "start": 0.0,
                "end": 0.0,
                "text": f"[Error: {str(e)}]"
            }]


class DiarizationModel:
    def __init__(self):
        pass

    def diarize(self, audio_path):
        return []


class SummarizationModel:
    def __init__(self):
        self.tokenizer, self.model = load_summarizer_elements()


    def summarize(self, text):
        print(f"Summarizing text (length: {len(text) if text else 0})...")
        if not text or len(text) < 20:
            return "Text too short to summarize."

        try:
            # Tokenize
            inputs = self.tokenizer(
                [text],
                max_length=1024,
                truncation=True,
                return_tensors="pt"
            )


            print("Generating summary...")
            with torch.no_grad():
                summary_ids = self.model.generate(
                    inputs["input_ids"],
                    attention_mask=inputs.get("attention_mask"),
                    num_beams=4,
                    max_length=130,
                    min_length=30,
                    early_stopping=True
                )

            summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            print(f"Summary generated: {summary[:50]}...")
            return summary
        except Exception as e:
            print(f"Summarization error: {e}")
            import traceback
            traceback.print_exc()
            return f"Error summarizing: {str(e)}"


def merge_transcript_diarization(transcript, diarization):
    if not transcript:
        return []

    merged = []
    current_speaker = "Speaker 1"
    for seg in transcript:
        merged.append({
            "start": seg['start'],
            "end": seg['end'],
            "speaker": current_speaker,
            "text": seg['text']
        })
    print(f"Merged {len(merged)} segments.")
    return merged

Overwriting model.py


In [ ]:
%%writefile processing.py
import threading
import queue
import time
import os
from model import TranscriptionModel, DiarizationModel, SummarizationModel, merge_transcript_diarization


class ProcessingPipeline:
    def __init__(self):
        self.audio_queue = queue.Queue()
        self.results = []
        self.is_running = False
        self.processing_thread = None
        self.final_summary = None

        # Initialize models
        self.transcriber = TranscriptionModel()
        self.diarizer = DiarizationModel()
        self.summarizer = SummarizationModel()


    def start(self):
        self.is_running = True
        self.processing_thread = threading.Thread(target=self._process_queue)
        self.processing_thread.start()


    def stop(self):
        self.is_running = False
        if self.processing_thread:
            self.processing_thread.join()

        # Trigger final summarization after processing stops
        self.generate_final_summary()


    def add_audio(self, audio_path):
        self.audio_queue.put(audio_path)


    def _process_queue(self):
        while self.is_running or not self.audio_queue.empty():
            try:
                # Get audio file from queue with a timeout to allow checking is_running
                audio_path = self.audio_queue.get(timeout=1)
                self._process_single_file(audio_path)
                self.audio_queue.task_done()
            except queue.Empty:
                continue


    def _process_single_file(self, audio_path):
        print(f"Processing {audio_path}...")
        try:
            # 1. Transcribe
            transcript = self.transcriber.transribe(audio_path)
            if not transcript:
                print("Warning: Transcription returned empty.")

            # 2. Diarize
            diarization = self.diarizer.diarize(audio_path)

            # 3. Merge
            merged_output = merge_transcript_diarization(transcript, diarization)

            # Store results
            self.results.extend(merged_output)
            print("File processed successfully.")

        except Exception as e:
            print(f"Error processing file: {e}")
            # Add error as a 'result' so user sees it
            self.results.append({
                "start": 0, "end": 0, "speaker": "System Error",
                "text": f"Processing Failed: {str(e)}"
            })


    def generate_final_summary(self):
        if not self.results:
            self.final_summary = "No content to summarize."
            return


        # Combine all text for summarization
        full_text = " ".join([f"{item['speaker']}: {item['text']}" for item in self.results])

        # 4. Summarize
        self.final_summary = self.summarizer.summarize(full_text)
        print("Final summary generated.")


    def get_results(self):
        return self.results


    def get_summary(self):
        return self.final_summary

Overwriting processing.py


In [ ]:
%%writefile app.py
import streamlit as st
import time
import os
from processing import ProcessingPipeline
from audiorecorder import audiorecorder


st.set_page_config(page_title="Live STT + Diarization + Summary", layout="wide")
st.title("🎙️ Transcription & Summarization Hub")


# CSS for better aesthetics
st.markdown("""
    <style>
    .stButton>button {
        width: 100%;
        border-radius: 5px;
        height: 3em;
    }
    .status-box {
        padding: 10px;
        border-radius: 5px;
        margin-bottom: 10px;
    }
    </style>
    """, unsafe_allow_html=True)


# Initialize Pipeline
if 'pipeline' not in st.session_state:
    with st.spinner("Loading AI Models (Whisper & Bart-CNN)... This may take 1-2 minutes on first run. Please wait. ☕"):
        st.session_state.pipeline = ProcessingPipeline()
        st.success("Models Loaded Successfully!")


# Tabs for Modes
tab1, tab2 = st.tabs(["🎙️ Choice 1: Live Recording", "📂 Choice 2: Upload Audio File"])


# --- MODE 1: LIVE RECORDING ---
with tab1:
    st.header("Live Recording Session")
    st.info("Record audio -> Process -> Diarize -> Summarize")

    col_rec, col_res = st.columns([1, 2])

    with col_rec:
        st.subheader("Recorder")
        # Audio Recorder Component
        audio = audiorecorder("Click to Record", "Click to Stop Recording")

        if len(audio) > 0:
            # When recording stops, save and process
            st.success(f"Recording finished: {audio.duration_seconds} seconds")

            # Save to file
            ts = int(time.time())
            filename = f"live_rec_{ts}.wav"
            audio.export(filename, format="wav")

            # Requested Feature: Audio Playback
            st.audio(filename)

            if st.button("Process Recording"):
                with st.spinner("Processing Audio... (Transcribing & Summarizing)"):
                    # Reset pipeline for fresh process
                    st.session_state.pipeline = ProcessingPipeline()
                    st.session_state.pipeline.start()

                    st.session_state.pipeline.add_audio(filename)
                    # Stop will wait for the thread to finish processing
                    st.session_state.pipeline.stop()

                st.rerun()


    with col_res:
        st.subheader("Results")
        results = st.session_state.pipeline.get_results()
        summary = st.session_state.pipeline.get_summary()


        if results:
            # Prepare Text for Download
            full_transcript_text = "\n".join([f"[{item['speaker']}]: {item['text']}" for item in results])

            st.download_button(
                label="⬇️ Download Diarized Transcript",
                data=full_transcript_text,
                file_name="diarized_transcript.txt",
                mime="text/plain"
            )


            with st.expander("Detailed Transcript", expanded=True):
                for item in results:
                    speaker_color = "blue" if item['speaker'] == "SPEAKER_00" else "green"
                    st.markdown(f"**:{speaker_color}[{item['speaker']}]:** {item['text']}")
                    st.caption(f"_{item['start']}s - {item['end']}s_")

        if summary:
            st.success("Summary Generated!")
            st.markdown(f"### Summary\n{summary}")
            st.download_button(
                label="⬇️ Download Summary",
                data=summary,
                file_name="summary.txt",
                mime="text/plain"
            )
        elif results and "System Error" in str(results):
             st.error("Processing failed. See transcript for details.")


# --- MODE 2: UPLOAD AUDIO FILE ---
with tab2:
    st.header("Upload Audio File")
    st.info("Upload -> Diarize -> Summarize -> Download")


    uploaded_file = st.file_uploader("Choose an audio file", type=['wav', 'mp3', 'm4a'])

    if uploaded_file:
        if st.button("Analyze File"):
            with st.spinner("Analyzing file..."):
                # Save temp
                temp_path = f"upload_{uploaded_file.name}"
                with open(temp_path, "wb") as f:
                    f.write(uploaded_file.getbuffer())

                # Clear previous state
                st.session_state.pipeline = ProcessingPipeline()
                st.session_state.pipeline.start()


                # Process
                st.session_state.pipeline.add_audio(temp_path)
                st.session_state.pipeline.stop() # Wait for finish

                st.toast("Analysis Complete!")
                st.rerun()


    # Display Results for Upload Mode
    summary_up = st.session_state.pipeline.get_summary()
    results_up = st.session_state.pipeline.get_results()


    if results_up:
         # Prepare Text for Download
        full_transcript_text_up = "\n".join([f"[{item['speaker']}]: {item['text']}" for item in results_up])

        st.divider()
        st.markdown("### 📄 Analysis Results")

        col_d1, col_d2 = st.columns(2)
        with col_d1:
             st.download_button(
                label="⬇️ Download Diarized Transcript",
                data=full_transcript_text_up,
                file_name="uploaded_diarized_transcript.txt",
                mime="text/plain"
            )

        if summary_up:
            st.markdown(f"**Summary:**\n\n{summary_up}")
            with col_d2:
                st.download_button(
                    label="⬇️ Download Summary",
                    data=summary_up,
                    file_name="uploaded_summary.txt",
                    mime="text/plain"
                )

        with st.expander("View Full Transcript"):
             for item in results_up:
                st.write(f"[{item['speaker']}] {item['text']}")

Overwriting app.py


In [ ]:
from google.colab import userdata

try:
    token = userdata.get('NGROK_AUTH_TOKEN')
    if token:
        print("✅ NGROK_AUTH_TOKEN is successfully set in Colab Secrets.")
    else:
        print("⚠️ NGROK_AUTH_TOKEN is set, but its value is empty or not accessible.")
except userdata.TimeoutException:
    print("❌ Timeout: NGROK_AUTH_TOKEN could not be retrieved. Ensure you are running this in the Colab UI and the secret is correctly configured.")
except userdata.SecretNotFoundError:
    print("❌ NGROK_AUTH_TOKEN not found in Colab Secrets. Please add it using the key icon (🔑) on the left sidebar.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

✅ NGROK_AUTH_TOKEN is successfully set in Colab Secrets.


In [ ]:
from pyngrok import ngrok
from google.colab import userdata
import subprocess
import time

# Kill any existing processes
!pkill -9 streamlit
!pkill -9 ngrok

# Securely retrieve ngrok token
ngrok.set_auth_token(userdata.get('NGROK_AUTH_TOKEN'))

# IMPORTANT: Start Streamlit FIRST and wait for it to be ready
print("🚀 Starting Streamlit...")
process = subprocess.Popen(
    ["python", "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait longer for Streamlit to fully start (critical step)
print("⏳ Waiting for Streamlit to start (15 seconds)...")
time.sleep(15)

# NOW create ngrok tunnel AFTER Streamlit is running
print("🌐 Creating ngrok tunnel...")
public_url = ngrok.connect(8501)

print("=" * 50)
print("✅ SUCCESS! Your app is running at:")
print(f"🔗 {public_url}")
print("=" * 50)
print("⚠️  Keep this cell running - stopping it will close the app.")
print("⚠️  Wait 5-10 seconds after clicking the URL for models to load.")

🚀 Starting Streamlit...
⏳ Waiting for Streamlit to start (15 seconds)...
🌐 Creating ngrok tunnel...
✅ SUCCESS! Your app is running at:
🔗 NgrokTunnel: "https://mistilled-unimported-milagros.ngrok-free.dev" -> "http://localhost:8501"
⚠️  Keep this cell running - stopping it will close the app.
⚠️  Wait 5-10 seconds after clicking the URL for models to load.
